# 03 - Train Flood Extent SegmentationTrains a U-Net on Sentinel-1 SAR imagery to segment flooded areas. Requires preprocessed SAR + mask GeoTIFFs in `data/processed/` (see `src/data/download_sentinel.py` and note on generating training masks below).

## A note on training masks for this stageUnlike xBD, there's no off-the-shelf labeled flood-mask dataset for Nepal specifically. Practical options, roughly in order of effort:1. Use an existing labeled SAR flood dataset (e.g. Sen1Floods11 - a public, labeled Sentinel-1 flood segmentation dataset) to pretrain/validate your U-Net, THEN run inference (unsupervised) on your Nepal AOI.2. Generate weak labels for the Nepal AOI via simple SAR thresholding/change-detection heuristics (water has distinctly low backscatter), and treat your trained model's Nepal predictions as an unsupervised case study rather than a supervised-tested result.Option 1 is the standard approach and is what most published flood-mapping papers do - recommend starting there.

## 1. Setup

In [ ]:
!git clone https://github.com/<YOUR_USERNAME>/flood-damage-assessment.git%cd flood-damage-assessment!pip install -q -r requirements.txtfrom google.colab import drivedrive.mount('/content/drive')

## 2. Sanity-check the model builds correctly

In [ ]:
!python -m src.models.flood_segmentation

## 3. Train (once data/processed/sar and data/processed/masks are populated)

In [ ]:
import yamlfrom src.train.train_segmentation import trainwith open('configs/config.yaml') as f:    config = yaml.safe_load(f)train(config)

## 4. Copy checkpoint to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/flood_project_data/checkpoints!cp checkpoints/flood_segmentation/best_model.pt /content/drive/MyDrive/flood_project_data/checkpoints/

## 5. Run inference on your Nepal case study SAR scenes

In [ ]:
# Load the pre/post Sentinel-1 composites exported via src/data/download_sentinel.py# and run predictor.predict() on the post-flood scene to get your flood mask.from src.inference.predict import FloodExtentPredictorpredictor = FloodExtentPredictor(checkpoint_path='checkpoints/flood_segmentation/best_model.pt')# result = predictor.predict(sar_tensor)# print('Flooded area fraction:', result['flooded_area_fraction'])